# Learning Curves

Notebook version of the learning-curve analyses you flagged: the training-level colored learning curves with one colorbar, plus the time-in-level summaries. Dataset and view selection follow the same pattern as the recent comparison notebooks.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "DailyMerge.py").exists() and (candidate / "DataFiles").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the Mafalda_analysis repo root from the current working directory."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

ROOT

## 2. Choose Datasets

`DATASET_SELECTIONS` is the safest option when mixing lines/cohorts. Set it to `None` to use all combinations of `LINES` and `COHORTS`. Use `ANIMAL_SELECTION` for a one-animal review.

In [ ]:
LINES = ["CNTNAP2"]
COHORTS = ["cohort3"]

# Explicit selections can mix lines/cohorts, e.g.:
# DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("CNTNAP2", "cohort3"), ("SHANK3", "cohort1")]
DATASET_SELECTIONS = None

# Example: "ASD0033" for a single animal, or None for group comparisons.
ANIMAL_SELECTION = None

## 3. Choose Comparison

Common presets:

- Compare genotypes within selected datasets: `COMPARISON = "genotypes"`, `SPLIT_BY = "none"`
- Keep cohorts/datasets separate within genotype: `COMPARISON = "genotypes"`, `SPLIT_BY = "dataset"`
- Compare the same genotype across datasets: `COMPARISON = "datasets"`, `GENOTYPES = ["hom"]`
- One view per animal: `COMPARISON = "animals"`
- Fully custom: `COMPARISON = "custom"` and edit `CUSTOM_SPECS`.

In [ ]:
COMPARISON = "genotypes"  # "genotypes", "datasets", "lines", "cohorts", "animals", or "custom"
SPLIT_BY = "none"         # for COMPARISON="genotypes": "none", "dataset", "line", or "cohort"
GENOTYPES = ["wt", "het", "hom"]

CUSTOM_SPECS = [
    # {"name": "CNTNAP2 hom all cohorts", "line": "CNTNAP2", "genotype": "hom"},
]

## 4. Load Data And Build Views

Rerun this if you change dataset selection, comparison mode, genotype selection, animal selection, or custom specs.

In [ ]:
from GroupComparison.config import ViewSpec
from Pipeline.group_comparison import build_group_views, load_groupcomparison_data, summarize_views
from Pipeline.learning_curves import build_view_colors, build_view_labels

data = load_groupcomparison_data(
    lines=LINES,
    cohorts=COHORTS,
    dataset_selections=DATASET_SELECTIONS,
)
df_plot = data["df_plot"]

if ANIMAL_SELECTION is not None:
    animal_id = str(ANIMAL_SELECTION).strip()
    df_plot = df_plot[df_plot["animal"].astype(str).str.strip() == animal_id].copy()
    if df_plot.empty:
        raise ValueError(f"No rows found for ANIMAL_SELECTION={animal_id!r} after dataset filters.")

if COMPARISON == "animals":
    animals = sorted(df_plot["animal"].dropna().astype(str).str.strip().unique())
    views = [
        ViewSpec(animal, lambda d, _animal=animal: d[d["animal"].astype(str).str.strip() == _animal].copy())
        for animal in animals
    ]
elif COMPARISON == "custom" and CUSTOM_SPECS:
    views = build_group_views(
        df_plot,
        comparison="custom",
        custom_specs=CUSTOM_SPECS,
    )
else:
    views = build_group_views(
        df_plot,
        comparison=COMPARISON,
        split_by=SPLIT_BY,
        genotypes=GENOTYPES,
    )

view_labels = build_view_labels(views)
view_colors = build_view_colors(df_plot, views)

print("Loaded datasets:", data["selections"])
print("Usable dataset keys:", data["usable_dataset_names"])
display(summarize_views(df_plot, views))
print("View labels:", view_labels)
print("View colors:", view_colors)

## 5. Prepare Once

This is the expensive step. Rerun it if you change filters, smoothing, normalized point count, time column, selected views, or datasets. You do not need to rerun it just to change `PLOT_MODE` below.

In [ ]:
import importlib
import Pipeline.learning_curves as lc
lc = importlib.reload(lc)

SPAN = 25
NORMALIZED_POINTS = 100
DROP_REPEAT_TRIALS = True
VALID_SUCCESS_VALUES = (1, -1)
TIME_COLUMN = "tared_trial_start"
LEVEL_MAX_EXCLUSIVE = 16

style = lc.default_style()

bundle = lc.prepare_learning_curve_bundle(
    df=df_plot,
    views=views,
    span=SPAN,
    normalized_points=NORMALIZED_POINTS,
    drop_repeat_trials=DROP_REPEAT_TRIALS,
    valid_success_values=VALID_SUCCESS_VALUES,
    time_col=TIME_COLUMN,
    level_max_exclusive=LEVEL_MAX_EXCLUSIVE,
    style=style,
    view_labels=view_labels,
    view_colors=view_colors,
)

print("Prepared rows:", len(bundle["df"]))
print("Training level range:", bundle["training_level_range"])

## 6. Plot From Prepared Data

Change `PLOT_MODE` and rerun this cell without recomputing preparation.

- `training_level_colormap`: one panel per selected view; one colorbar for training level
- `time_in_level_scatter`: per-animal time in level with jittered points
- `time_in_level_boxplot`: grouped boxplots by training level
- `all`: make all figure families

In [ ]:
PLOT_MODE = "training_level_colormap"
PLOT_VIEW = None     # set to a view name to plot just one selected group/view
TIME_UNIT = "hour"  # "sec", "min", or "hour"

plot_views = [v for v in views if PLOT_VIEW is None or v.name == PLOT_VIEW]
if not plot_views:
    raise ValueError(f"PLOT_VIEW={PLOT_VIEW!r} did not match any selected view.")

out = lc.plot_learning_curve_figures(
    bundle=bundle,
    views=plot_views,
    plot_mode=PLOT_MODE,
    time_unit=TIME_UNIT,
    show=True,
)

out["figures"]

## 7. Example Recipes

### Compare genotypes within one cohort

```python
DATASET_SELECTIONS = [("CNTNAP2", "cohort3")]
COMPARISON = "genotypes"
SPLIT_BY = "none"
GENOTYPES = ["wt", "het", "hom"]
```

### One learning-curve figure per animal

```python
COMPARISON = "animals"
PLOT_MODE = "training_level_colormap"
```

### Compare only the hom group across datasets

```python
DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("CNTNAP2", "cohort3"), ("SHANK3", "cohort1")]
COMPARISON = "datasets"
GENOTYPES = ["hom"]
```

### Time in level as boxplots

```python
PLOT_MODE = "time_in_level_boxplot"
TIME_UNIT = "hour"
```